# Bronze → Silver

Conceitos praticados:
- **Padronização**: todas as colunas em português e com a tipagem correta
- **Deduplicação** por chave de negócio
- **Conversão segura de tipos**: textos fora de contexto (Column Shift) viram `NULL` sem interromper o pipeline
- **Regras de domínio**: escala de notas, valores negativos, status válidos

As tabelas Bronze são apenas lidas; nenhuma alteração é feita nelas.

In [0]:
catalog = "cinedata_analytics"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")

DataFrame[]

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Datas que não casam com o padrão testado viram NULL em vez de gerar erro de versão do parser
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

def to_double(col_name: str):
    """Conversão segura: só converte quando o texto é um número, caso contrário retorna NULL."""
    valor = F.trim(F.col(col_name))
    return F.when(valor.rlike(r"^-?\d+(\.\d+)?$"), valor.cast("double"))

def to_int(col_name: str):
    """Conversão segura para inteiro: valores com texto ou casas decimais viram NULL."""
    valor = F.trim(F.col(col_name))
    return F.when(valor.rlike(r"^-?\d+$"), valor.cast("int"))

## 1. tb_info_filmes
- **Deduplicação**: uma linha por filme, mantendo a versão mais recente pela `ingestion_datetime`
- **Status**: antes da tradução, a coluna é normalizada (caixa baixa, hífens e ruídos viram espaço). Valores corrompidos ou não mapeáveis viram `"Não Informado"`
- **Data multi-formato**: a origem mistura `yyyy-MM-dd`, `dd/MM/yyyy` e `MM-dd-yyyy`. Testamos os três padrões e só fica `NULL` o que não converte em nenhum deles
- Coluna derivada `ano_lancamento`

In [0]:
df_bronze_info = spark.table(f"{bronze_schema}.tb_movies_info")

# Deduplicação: a Bronze é append, então mantemos apenas a versão mais recente de cada filme
janela_mais_recente = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_silver_info = (
    df_bronze_info
    .withColumn("rn", F.row_number().over(janela_mais_recente))
    .filter(F.col("rn") == 1)
    # Normaliza o status antes de traduzir: caixa baixa e ruídos/hífens viram espaço
    .withColumn("status", F.trim(F.regexp_replace(F.lower(F.col("status")), r"[^a-z]+", " ")))
    .withColumn("status",
        F.when(F.col("status") == "released", "Lançado")
        .when(F.col("status") == "post production", "Pós-Produção")
        .when(F.col("status") == "in production", "Em Produção")
        .when(F.col("status") == "planned", "Planejado")
        .when(F.col("status") == "rumored", "Rumores")
        .when(F.col("status") == "canceled", "Cancelado")
        .otherwise("Não Informado")  # corrompidos ou não mapeáveis
    )
    # Data multi-formato: testa os três padrões da origem; só fica NULL o que não converte em nenhum
    .withColumn("release_date", F.coalesce(
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("yyyy-MM-dd"))),
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("dd/MM/yyyy"))),
        F.to_date(F.try_to_timestamp(F.col("release_date"), F.lit("MM-dd-yyyy"))),
    ))
    .withColumn("runtime", to_int("runtime"))
    .select("id", "title", "original_title", "release_date", "runtime",
            "original_language", "status", "overview", "tagline")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

df_silver_info.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_info_filmes")

display(df_silver_info.limit(10))

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,42,en,Lançado,"No one, and nothing, is off-limits for comedian Aisha Brown, as she takes on her boyfriend’s penis, racism, clinical depression, and Donald Trump, in this hilarious one-hour comedy special from Just For Laughs.",null,2020
1000075,Largo Winch III : Le prix de l'argent,Largo Winch III : Le prix de l'argent,2024-10-16,0,fr,Em Produção,null,null,2024
1000486,THE PINK JACKET,O Casaco Rosa,2022-07-15,8,pt,Lançado,"A political musical film about a Pink Jacket that always has something up its sleeve. In the comfort of its home, Pink Jacket uses couture and torture to stitch up the opponents of the system. Mónica Santos has already won twice the Audience Award at Curtas with “Between the Shadows” (2018) and “Amélia & Duarte” (2015).",null,2022
1000572,A Great Friend,Les Choses simples,2023-02-22,95,fr,Lançado,"Vincent is a famous successful entrepreneur. One day, a car breakdown on a mountain road temporarily interrupts his frantic lifestyle. Pierre, who lives apart from the modern world in the middle of sublime nature, comes to his aid and offers him his hospitality. The meeting between these two men, who are opposites will upset their respective beliefs and they will find themselves laughing. The question is...deep down, are they really each living the lives they want to live?",null,2023
1000593,Taniel,Taniel,2018-03-29,20,en,Lançado,"\""""\""""Taniel""""\"""" looks at the last months of poet Taniel Varoujan's life""""/e4w0H7RKwwYz5hzQYnmQqesWOfo.jpg",null,2018
1000618,To The North,Spre nord,2022-08-31,122,ro,Lançado,null,null,2022
1000622,Elvis and the USS Arizona,Elvis and the USS Arizona,2021-11-14,58,en,Lançado,The surprising story of how one of music's biggest icons helped to establish a USS Arizona memorial in Pearl Harbor. Elvis’ fundraising concert drew public attention to the plight and helped to galvanize efforts to finish the USS Arizona Memorial as it stands today.,The story of how one of music's biggest icons helped to establish a USS Arizona memorial.,2021
1000948,A Monstrous Marriage,Un matrimonio mostruoso,2023-06-21,101,it,Lançado,"Vladimiro and Brunilde are grappling with the disappearance of Nando, the head of the family. The sad event brought together the human family of Luna, daughter of Nando, and the monstrous one of her husband Adalberto. Humans, vampires, witches, werewolves and ghosts will find themselves in the throes of a new monstrous marriage...",null,2023
1001017,"Shoot faster, baby!","Shoot faster, baby!",2017-12-28,12,en,Lançado,"Amanda dreams of getting rid of her husband and persuades Mike to help her. He agrees, and with this he signs his own death warrant...",null,2017
1001209,Detroit Dreams,Detroit Dreams,2022-06-05,94,en,Lançado,"Music Promoter Bobby trapped in the middle of a rival gangs beef, while trying to put Detroit Music scene on his back.","Our music, Our story",2022


## 2. tb_financeiro_filmes
- Textos que representam ausência (`N/A`, `Unknown`, `Não Informado`) viram `NULL` antes da conversão
- Higienização de símbolos de moeda (`$`, `USD`), pontuações de milhar e abreviações de escala (K = mil, M = milhão, B = bilhão; ex.: `200.0M` → 200.000.000)
- Conversão para `DECIMAL(18,2)`; valores zerados ou negativos são tratados como ausentes
- Valores em BRL calculados com a cotação extraída da API (a mais recente gravada em `bronze.tb_cotacao_dolar`)
- **Lucro** só é calculado quando orçamento e receita existem; caso contrário fica `NULL`. A **margem** só é calculada com receita positiva, evitando divisão por zero

In [0]:
cotacao_dolar = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .first()["cotacaoCompra"]
)
print(f"Cotação utilizada: R$ {cotacao_dolar}")

textos_ausentes = ["N/A", "UNKNOWN", "NÃO INFORMADO"]

def limpar_valor_monetario(col_name: str):
    valor = F.upper(F.trim(F.col(col_name)))
    # Textos de ausência viram NULL antes da conversão
    valor = F.when(valor.isin(textos_ausentes), None).otherwise(valor)
    # Remove símbolos de moeda e separadores de milhar
    valor = F.regexp_replace(valor, r"USD|\$|,|\s", "")

    # Abreviações de escala: K = mil, M = milhão, B = bilhão (ex.: 200.0M = 200.000.000)
    multiplicador = (
        F.when(valor.endswith("K"), 1000)
        .when(valor.endswith("M"), 1000000)
        .when(valor.endswith("B"), 1000000000)
        .otherwise(1)
    )
    numero = F.regexp_replace(valor, r"[KMB]$", "")

    # Limite de dígitos para o resultado caber em DECIMAL(18,2) sem estourar
    numero_valido = (
        F.when(valor.rlike(r"[KMB]$"), numero.rlike(r"^-?\d{1,7}(\.\d+)?$"))
        .otherwise(numero.rlike(r"^-?\d{1,16}(\.\d+)?$"))
    )
    return F.when(numero_valido, (numero.cast("decimal(18,2)") * multiplicador).cast("decimal(18,2)"))

Cotação utilizada: R$ 5.0435


In [0]:
df_bronze_financeiro = spark.table(f"{bronze_schema}.tb_movies_financials")

df_silver_financeiro = (
    df_bronze_financeiro
    .withColumn("budget", limpar_valor_monetario("budget"))
    .withColumn("revenue", limpar_valor_monetario("revenue"))
    # Valores zerados ou negativos são tratados como ausentes
    .withColumn("budget", F.when(F.col("budget") > 0, F.col("budget")))
    .withColumn("revenue", F.when(F.col("revenue") > 0, F.col("revenue")))
    .select("id", "budget", "revenue")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
    # Conversão para Reais com a cotação extraída da API
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.lit(cotacao_dolar).cast("decimal(10,4)")).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * F.lit(cotacao_dolar).cast("decimal(10,4)")).cast("decimal(18,2)"))
    # Lucro só existe com orçamento e receita; ausência não vira prejuízo
    .withColumn("lucro_usd",
        F.when(F.col("orcamento_usd").isNotNull() & F.col("receita_usd").isNotNull(),
               F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl",
        F.when(F.col("orcamento_brl").isNotNull() & F.col("receita_brl").isNotNull(),
               F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))
    # Margem só com receita positiva, evitando divisão por zero
    .withColumn("margem_lucro_percentual",
        F.when(F.col("lucro_usd").isNotNull() & (F.col("receita_usd") > 0),
               F.round(F.col("lucro_usd") / F.col("receita_usd") * 100, 2)).cast("double"))
)

df_silver_financeiro.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes")

display(df_silver_financeiro.limit(10))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
293660,58000000.00,null,292523000.00,null,null,null,null
299536,300000000.00,2052415039.00,1513050000.00,10351355249.20,1752415039.00,8838305249.20,85.38
299534,356000000.00,2800000000.00,1795486000.00,14121800000.00,2444000000.00,12326314000.00,87.29
475557,55000000.00,1074458282.00,277392500.00,5419030345.27,1019458282.00,5141637845.27,94.88
271110,250000000.00,null,1260875000.00,null,null,null,null
284054,200000000.00,1349926083.00,1008700000.00,6808352199.61,1149926083.00,5799652199.61,85.18
284052,180000000.00,676343174.00,907830000.00,3411136798.07,496343174.00,2503306798.07,73.39
315635,175000000.00,880166924.00,882612500.00,4439121881.19,705166924.00,3556509381.19,80.12
283995,200000000.00,863756051.00,1008700000.00,4356353643.22,663756051.00,3347653643.22,76.85
297761,175000000.00,746846894.00,882612500.00,3766722309.89,571846894.00,2884109809.89,76.57


## 3. tb_metricas_engajamento
- **Popularidade**: a origem mistura vírgula e ponto como separador decimal (`154,34` e `72.735`). A vírgula é trocada por ponto antes da conversão, para que o valor não vire `NULL`
- **Conversão segura**: textos deslocados (Column Shift) nas colunas de notas e votos viram `NULL`
- **Regras de negócio**: notas fora do intervalo 0 a 10 (incluindo erros de escala) e contagens ou popularidade negativas viram `NULL`

In [0]:
df_bronze_metricas = spark.table(f"{bronze_schema}.tb_movies_metrics")

df_silver_metricas = (
    df_bronze_metricas
    # Separador decimal inconsistente: vírgula vira ponto para o valor não ser invalidado
    .withColumn("popularity", F.regexp_replace(F.trim(F.col("popularity")), ",", "."))
    # Linha com Column Shift (texto nas colunas do IMDb): um inteiro na popularidade é um ano fora de contexto
    .withColumn("popularity",
        F.when((F.col("averageRating").rlike("[A-Za-z]") | F.col("numVotes").rlike("[A-Za-z]"))
               & F.col("popularity").rlike(r"^\d+$"), None)
        .otherwise(F.col("popularity")))
    # Conversão segura: textos deslocados pelo Column Shift viram NULL sem interromper o pipeline
    .withColumn("popularidade", to_double("popularity"))
    .withColumn("nota_media_tmdb", to_double("vote_average"))
    .withColumn("qtd_votos_tmdb", to_int("vote_count"))
    .withColumn("nota_media_imdb", to_double("averageRating"))
    .withColumn("qtd_votos_imdb", to_int("numVotes"))
    # Regras de negócio: notas fora de 0 a 10 e contagens ou popularidade negativas viram NULL
    .withColumn("popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")))
    .withColumn("nota_media_tmdb", F.when(F.col("nota_media_tmdb").between(0, 10), F.col("nota_media_tmdb")))
    .withColumn("nota_media_imdb", F.when(F.col("nota_media_imdb").between(0, 10), F.col("nota_media_imdb")))
    .withColumn("qtd_votos_tmdb", F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb")))
    .withColumn("qtd_votos_imdb", F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb")))
    .select(F.col("id").alias("id_filme"), "popularidade", "nota_media_tmdb",
            "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb")
)

df_silver_metricas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento")

display(df_silver_metricas.limit(10))

id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
293660,72.735,7.606,28894,8.0,1270339
299536,154.34,8.255,27713,8.4,1406782
299534,91.756,8.263,23857,8.4,1484150
475557,54.522,8.168,23425,8.3,1723035
271110,70.741,7.4,21541,7.8,947222
284054,43.665,7.39,null,7.3,924922
284052,70.535,7.427,20935,7.5,895880
315635,65.88,7.345,20507,7.4,835116
283995,67.553,7.624,20353,7.6,844767
297761,35.356,5.909,20097,5.9,null


## 4. tb_avaliacoes_usuarios
- Remoção de registros integralmente duplicados (mesmo filme, usuário, nota e comentário)
- `nota_usuario` fora da escala de 0 a 10 é descartada e vira `NULL`
- Comentários vazios ou só com espaços recebem o texto padronizado `"Sem comentário"`

In [0]:
df_bronze_avaliacoes = spark.table(f"{bronze_schema}.tb_movies_reviews")

df_silver_avaliacoes = (
    df_bronze_avaliacoes
    # Remove avaliações integralmente duplicadas (filme, usuário, nota e comentário)
    .dropDuplicates(["id", "nome", "nota", "comentario"])
    .withColumn("nota", to_double("nota"))
    # Nota fora da escala de 0 a 10 é descartada
    .withColumn("nota", F.when(F.col("nota").between(0, 10), F.col("nota")))
    # Comentário vazio ou só com espaços recebe o texto padronizado
    .withColumn("comentario",
        F.when(F.trim(F.coalesce(F.col("comentario"), F.lit(""))) == "", "Sem comentário")
        .otherwise(F.col("comentario")))
    .select("id", "nome", "nota", "comentario")
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

df_silver_avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios")

display(df_silver_avaliacoes.limit(10))

id_filme,nome_usuario,nota_usuario,comentario_usuario
797814,Matheus Cavalcanti,1.1,Péssimo em todos os sentidos.
800797,Sérgio Almeida 335,7.2,Filme interessante com boas atuações.
1086795,Rodrigo Oliveira 273,9.7,Excepcional! História envolvente e atuações impecáveis.
1054508,Tatiana Oliveira 369,7.7,Legal! Uma boa opção para o fim de semana.
776552,Letícia Lopes,8.4,Filme interessante com boas atuações.
750569,Pedro Souza 611,4.4,Sem comentário
464175,Mônica Cavalcanti,7.7,Filme interessante com boas atuações.
823195,Rodrigo Mendes,2.9,Péssimo em todos os sentidos.
1055332,Gustavo Dias 303,null,Sem comentário
679864,Lucas Freitas 371,9.2,Perfeito! Um dos melhores filmes que já vi.


## 5. tb_generos
- Os separadores extras (`;` e `|`) viram vírgula antes do `split` + `explode`
- Cada linha passa a representar um único gênero por filme
- Resíduos do Column Shift (valores em branco, numéricos e textos descritivos) são removidos validando contra o domínio de gêneros do TMDB

In [0]:
df_bronze_creditos = spark.table(f"{bronze_schema}.tb_credits_and_tags")

# Domínio de gêneros do TMDB: tudo fora dele é resíduo do Column Shift
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary", "Drama",
    "Family", "Fantasy", "History", "Horror", "Music", "Mystery", "Romance",
    "Science Fiction", "TV Movie", "Thriller", "War", "Western",
]

df_silver_generos = (
    df_bronze_creditos
    # Padroniza os separadores (; e |) para vírgula antes do split
    .withColumn("genres", F.regexp_replace(F.col("genres"), r"[;|]", ","))
    .select(F.col("id").alias("id_filme"), F.explode(F.split(F.col("genres"), ",")).alias("nome_genero"))
    .withColumn("nome_genero", F.trim(F.col("nome_genero")))
    .filter(F.col("nome_genero").isin(generos_validos))
    .dropDuplicates(["id_filme", "nome_genero"])
)

df_silver_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_generos")

display(df_silver_generos.limit(10))

id_filme,nome_genero
353486,Comedy
399055,Romance
343668,Comedy
577922,Action
466282,Comedy
359724,Action
328387,Mystery
381283,Thriller
420809,Adventure
335787,Adventure


## 6. tb_pessoas_empresas
Tabela unificada com quatro tipos de entidade:

| Coluna origem | tipo_entidade |
|---|---|
| `cast` | Ator |
| `directors` | Diretor |
| `writers` | Roteirista |
| `production_companies` | Produtora |

- Mesmo tratamento de separadores (`;` e `|`) da tabela de gêneros, seguido de `split` + `explode`
- Remoção de resíduos do Column Shift: valores em branco ou marcadores de ausência (`N/A`, `Nenhum`, `[]`), valores numéricos (sem nenhuma letra) e textos descritivos (que começam com letra minúscula ou têm mais de 8 palavras, como trechos de sinopse e palavras-chave)
- Capitalização padronizada com `initcap` e eliminação de duplicados

In [0]:
colunas_entidades = {
    "cast": "Ator",
    "directors": "Diretor",
    "writers": "Roteirista",
    "production_companies": "Produtora",
}

marcadores_ausencia = ["", "N/A", "NENHUM", "[]"]

# Une as quatro colunas em uma tabela, categorizada pelo tipo de entidade
df_entidades = None
for coluna, tipo in colunas_entidades.items():
    df_tipo = (
        df_bronze_creditos
        .withColumn(coluna, F.regexp_replace(F.col(coluna), r"[;|]", ","))
        .select(F.col("id").alias("id_filme"), F.explode(F.split(F.col(coluna), ",")).alias("nome_entidade"))
        .withColumn("tipo_entidade", F.lit(tipo))
    )
    df_entidades = df_tipo if df_entidades is None else df_entidades.unionByName(df_tipo)

df_silver_pessoas_empresas = (
    df_entidades
    .withColumn("nome_entidade", F.trim(F.col("nome_entidade")))
    # Remove resíduos do Column Shift: ausências, valores numéricos e textos descritivos
    .filter(~F.upper(F.col("nome_entidade")).isin(marcadores_ausencia))
    .filter(F.col("nome_entidade").rlike(r"[A-Za-zÀ-ÿ]"))
    .filter(~F.col("nome_entidade").rlike(r"^[a-z]"))
    .filter(F.size(F.split(F.col("nome_entidade"), r"\s+")) <= 8)
    # Padroniza a capitalização e elimina duplicados
    .withColumn("nome_entidade", F.initcap(F.col("nome_entidade")))
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

df_silver_pessoas_empresas.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas")

display(df_silver_pessoas_empresas.limit(10))

id_filme,nome_entidade,tipo_entidade
496243,Song Kang-ho,Ator
335984,Carla Juri,Ator
515001,Rebel Wilson,Ator
515001,Alfie Allen,Ator
445571,Jason Bateman,Ator
381289,Luke Kirby,Ator
315635,Laura Harrier,Ator
339403,Cj Jones,Ator
330459,Alan Tudyk,Ator
246655,Sophie Turner,Ator


## 7. tb_cotacao_dolar
- **Série temporal contínua**: calendário do primeiro dia com cotação até a data de execução, com uma linha por dia (mantendo o lote mais recente quando a API é chamada mais de uma vez)
- **Forward Fill**: dias sem cotação (fins de semana e feriados) recebem o valor do último dia útil disponível

In [0]:
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")

# Uma cotação por dia: execuções repetidas da API geram lotes sobrepostos na Bronze
janela_lote_recente = Window.partitionBy("data_cotacao").orderBy(F.col("ingestion_datetime").desc())

df_cotacao_dia = (
    df_bronze_cotacao
    .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
    .withColumn("rn", F.row_number().over(janela_lote_recente))
    .filter(F.col("rn") == 1)
    .select("data_cotacao", "cotacao_compra")
)

# Calendário contínuo do primeiro dia com cotação até a data de execução
df_calendario = (
    df_cotacao_dia
    .agg(F.min("data_cotacao").alias("data_inicio"))
    .select(F.explode(F.sequence(F.col("data_inicio"), F.current_date())).alias("data_cotacao"))
)

# Forward Fill: fins de semana e feriados recebem a cotação do último dia útil
janela_forward_fill = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_silver_cotacao = (
    df_calendario
    .join(df_cotacao_dia, on="data_cotacao", how="left")
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_forward_fill))
    .orderBy("data_cotacao")
)

df_silver_cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar")

display(df_silver_cotacao)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_compra
2026-09-29,5.2198
2026-09-30,5.1803
2026-10-01,5.2073
2026-10-02,5.2232
2026-10-03,5.2232
2026-10-04,5.2232
2026-10-05,4.9853
2026-10-06,4.9692
2026-10-07,4.9929
2026-10-08,5.0113
